# LC 981 — Time Based Key-Value Store
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Binary Search
**Pattern:** HashMap of Sorted Lists + Binary Search Floor Query

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Store each key's
history as a list of (timestamp, value) pairs in
insertion order (timestamps always increase).
For a get query, binary search that list for the
largest timestamp that is ≤ the query timestamp.
</div>

## Official Problem Statement

Design a time-based key-value data structure that
can store multiple values for the same key at
different time stamps and retrieve the key's value
at a certain timestamp.

Implement the `TimeMap` class:

- `TimeMap()` Initialises the object.
- `set(key, value, timestamp)` Stores the value
  for key at the given timestamp.
- `get(key, timestamp)` Returns a value such that
  `set` was called previously with
  `timestamp_prev <= timestamp`. If there are
  multiple such values, return the one with the
  largest `timestamp_prev`. If no such value
  exists, return `""`.

**Example 1:**
```
Input:
  ["TimeMap","set","get","get","set","get","get"]
  [[],["foo","bar",1],["foo",1],["foo",3],
   ["foo","bar2",4],["foo",4],["foo",5]]
Output:
  [null, null, "bar", "bar", null, "bar2", "bar2"]
```

**Constraints:**
- `1 <= key.length, value.length <= 100`
- `1 <= timestamp <= 10^7`
- All timestamps in `set` are strictly increasing
- At most `2 * 10^5` calls to `set` and `get`

## What This Is Actually Asking

Build a key-value store that remembers history.
Each key can be set multiple times at different
timestamps. A get query asks: what was the value
for this key at or just before this timestamp?
Return the most recent value that was set at or
before the requested time.

## Walk Through an Example by Hand

```
set("foo", "bar",  t=1)  -> store["foo"] = [(1,"bar")]
set("foo", "bar2", t=4)  -> store["foo"] = [(1,"bar"),(4,"bar2")]

get("foo", t=1):
  list = [(1,"bar"),(4,"bar2")]
  bisect_right on timestamps [1,4] for t=1 -> pos=1
  pos-1 = 0 -> list[0] = (1,"bar")  answer: "bar"

get("foo", t=3):
  bisect_right on [1,4] for t=3 -> pos=1
  pos-1 = 0 -> list[0] = (1,"bar")  answer: "bar"
  (t=3 is between 1 and 4; last set before 3 is t=1)

get("foo", t=4):
  bisect_right on [1,4] for t=4 -> pos=2
  pos-1 = 1 -> list[1] = (4,"bar2")  answer: "bar2"

get("foo", t=5):
  bisect_right on [1,4] for t=5 -> pos=2
  pos-1 = 1 -> list[1] = (4,"bar2")  answer: "bar2"

get("foo", t=0):
  bisect_right on [1,4] for t=0 -> pos=0
  pos-1 = -1 -> no entry before t=0  answer: ""
```

## The Picture

```
store = {
  "foo": [(1, "bar"), (4, "bar2")]
}               ^
                timestamps always increase
                (constraint: set() called in order)

get("foo", t=3):  floor query — largest ts <= 3

Timestamps: [1,  4]
Query t=3:   ^      bisect_right([1,4], 3) = 1
                    pos-1 = 0 -> "bar"

Timeline:
t=0  t=1    t=2  t=3   t=4     t=5
 |    |      |    |     |       |
""  "bar"  "bar" "bar" "bar2" "bar2"

bisect_right gives insertion point AFTER equal
timestamps. Subtract 1 to get the floor.
If pos == 0 no valid entry exists -> return "".

Data structure summary:
  set:  O(1) — append to list
  get:  O(log n) — bisect on sorted timestamps
```

## When To Use This Pattern

- When you need the **most recent entry at or
  before a given time**, think **floor query —
  bisect_right minus one**
- When inserts are always in increasing timestamp
  order, think **append to list — already sorted**
- When `bisect_right` returns 0, think
  **no valid entry — return empty string**
- When each key has its own history, think
  **defaultdict(list) keyed by the string key**
- When asked to design a time-versioned store,
  think **HashMap of sorted (timestamp, value) lists**

## The Approach

In `__init__` create a defaultdict of lists.
In `set` append a (timestamp, value) tuple to the
list for the given key.
In `get` retrieve the list for the key (empty if
key not seen). Use bisect_right on the list's
timestamps to find the insertion position for the
query timestamp. Subtract one to get the floor
index. If that index is below zero return empty
string; otherwise return the value at that index.

In [ ]:
from bisect import bisect_right  # floor query on sorted list

In [ ]:
def test_harness(cls):
    """
    Replay sequences of (op, args, expected) against cls.
    ops: 'set' -> (key, value, ts),  expected ignored (None)
         'get' -> (key, ts),         expected is the return value
    """
    sequences = [
        # sequence 1 — from the problem statement
        [
            ("set", ("foo", "bar",  1), None),
            ("get", ("foo", 1),        "bar"),
            ("get", ("foo", 3),        "bar"),
            ("set", ("foo", "bar2", 4), None),
            ("get", ("foo", 4),        "bar2"),
            ("get", ("foo", 5),        "bar2"),
        ],
        # sequence 2 — timestamp before any set
        [
            ("set", ("a", "x", 5),  None),
            ("get", ("a", 4),       ""),
            ("get", ("a", 5),       "x"),
            ("get", ("a", 6),       "x"),
        ],
        # sequence 3 — unknown key
        [
            ("get", ("missing", 1), ""),
        ],
        # sequence 4 — exact timestamp match
        [
            ("set", ("k", "v1", 10), None),
            ("set", ("k", "v2", 20), None),
            ("get", ("k", 10),       "v1"),
            ("get", ("k", 20),       "v2"),
            ("get", ("k", 15),       "v1"),
            ("get", ("k", 25),       "v2"),
        ],
    ]

    passed = 0
    total = 0
    for s_i, seq in enumerate(sequences):
        obj = cls()
        seq_pass = True
        for op, args, expected in seq:
            if op == "set":
                obj.set(*args)
            else:
                result = obj.get(*args)
                total += 1
                ok = result == expected
                if not ok:
                    seq_pass = False
                    print(
                        f"  Seq {s_i+1} FAILED: "
                        f"get{args} expected "
                        f"{expected!r} got {result!r}"
                    )
        if seq_pass:
            passed += 1
            print(f"Sequence {s_i+1}: PASSED")
        else:
            print(f"Sequence {s_i+1}: FAILED")

    print(f"\n{passed}/{len(sequences)} sequences passed")

In [ ]:
class TimeMap:
    """
    Time-versioned key-value store.

    set: append (timestamp, value) to store[key] list.
    get: bisect_right on timestamps in store[key] to
    find floor position (pos-1). Return "" if pos==0,
    else return value at pos-1.

    set: O(1) — append
    get: O(log n) — binary search on timestamps list
    Space: O(total set calls)
    """

    def __init__(self):
        pass

    def set(self, key: str, value: str,
            timestamp: int) -> None:
        pass

    def get(self, key: str, timestamp: int) -> str:
        pass


# Quick debug — run this cell while building
tm = TimeMap()
tm.set("foo", "bar",  1)
print(tm.get("foo", 1))   # "bar"
print(tm.get("foo", 3))   # "bar"
tm.set("foo", "bar2", 4)
print(tm.get("foo", 4))   # "bar2"
print(tm.get("foo", 5))   # "bar2"

In [ ]:
# Uncomment and run when solution is ready
# test_harness(TimeMap)

## Complexity

| Operation | Time | Space |
|---|---|---|
| `set` | O(1) — list append | O(1) per call |
| `get` (linear scan) | O(n) | O(1) |
| `get` (binary search) | O(log n) | O(1) |
| Total space | — | O(total set calls) |

The key insight is that timestamps are strictly
increasing per the constraint, so the list is
always sorted — no explicit sort needed, and
binary search works immediately.

## Real World Connection

At Citi, the capacity planning system stores a
time-versioned configuration per server: each time
a server's SLA tier is updated, a new (timestamp,
tier) record is appended to that server's history.
When the Prophet forecasting pipeline queries
"what tier was server X in at time T?", it runs
a floor query — exactly TimeMap.get — to retrieve
the most recent tier set at or before T.
On AWS, DynamoDB's Time-to-Live and versioned item
history follow the same model: each item version
has a timestamp, and point-in-time reads perform
a binary floor lookup against the version index
instead of a full table scan.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra